# NQ-open Query Filtering

**Pipeline reference**: Step 2 — Query Filtering

**Goal**: filter `florin-hf/nq_open_gold` (Silvestri et al.) to keep only queries where:
1. Answers have ≤ 5 tokens
2. Both question and answer contain Wikidata entities recognized by ReFiNed

**Output**: filtered dataset with extracted entities, saved as Parquet for reuse in Step 3 (KG Subgraph Construction).

**Dataset**: `florin-hf/nq_open_gold` — NQ-open enriched with gold documents from `wiki_dump2018_nq_open`.

| Field | Type | Description |
|-------|------|-------------|
| `example_id` | int64 | Original NQ identifier |
| `question` | str | Question text |
| `answers` | List[str] | Correct answer(s) |
| `text` | str | Gold document text |
| `idx_gold_in_corpus` | int64 | Index in the HF corpus (not used — our retrieval is FAISS-based) |

## Step 0 — Configuration & Dataset Loading

In [1]:
from pathlib import Path
from huggingface_hub import login

# --- HuggingFace authentication (optional, speeds up downloads) ---
token_path = Path.cwd() / ".hf_token"
if token_path.exists():
    token = token_path.read_text().strip()
    if token and token != "YOUR_TOKEN_HERE":
        login(token=token)
        print("Logged in to HuggingFace Hub")
    else:
        print("No token set — put your HF token in .hf_token")
else:
    print("No .hf_token file found — running unauthenticated (slower downloads)")

Logged in to HuggingFace Hub


In [2]:
from datasets import load_dataset

# --- Load all splits ---
# florin-hf/nq_open_gold: NQ-open with gold documents from Silvestri et al.
# Columns: example_id, question, answers, text, idx_gold_in_corpus
dataset = load_dataset("florin-hf/nq_open_gold")

print(f"Splits: {list(dataset.keys())}")
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data):,} rows")

print(f"\nColumns: {dataset['train'].column_names}")
print(f"\nSample (train[0]):")
dataset["train"][0]

Splits: ['train', 'validation', 'test']
  train: 72,209 rows
  validation: 8,006 rows
  test: 2,889 rows

Columns: ['question', 'idx_gold_in_corpus', 'answers', 'text', 'example_id']

Sample (train[0]):


{'question': 'total number of death row inmates in the us',
 'idx_gold_in_corpus': 20970735,
 'answers': ['2,718'],
 'text': 'As of June 14 , 2018 , there were 2,718 death row inmates in the United States . The number of death row inmates changes daily with new convictions , appellate decisions overturning conviction or sentence alone , commutations , or deaths ( through execution or otherwise ) . Due to this fluctuation as well as lag and inconsistencies in inmate reporting procedures across jurisdictions , the information in this article may be out of date .',
 'example_id': -6802534628745605728}

In [3]:
import pandas as pd
from datasets import concatenate_datasets

# --- Merge all splits into one ---
# We filter across the entire dataset; split boundaries are irrelevant
# for our pipeline (we do our own retrieval + evaluation on all queries).
all_queries = concatenate_datasets([dataset[s] for s in dataset])
print(f"Total queries (all splits): {len(all_queries):,}")

Total queries (all splits): 83,104


## Step 1 — Token count filter (≤ 5 tokens)

We use the **Contriever tokenizer** (`facebook/contriever`, BERT-based) to count tokens.
Whitespace split would be simpler but imprecise: subword tokenization can split
proper nouns ("Calrissian" → ["cal", "##ris", "##sian"] = 3 tokens).

We also check for **split answers** — queries where some answer variants pass
the ≤ 5 threshold and others don't — to decide whether to filter on min, max, or all.

In [4]:
from transformers import AutoTokenizer

# --- Contriever tokenizer (BERT wordpiece) ---
tokenizer = AutoTokenizer.from_pretrained("facebook/contriever")

TOKEN_THRESHOLD = 5

# Count tokens for each answer variant (excluding special tokens [CLS], [SEP])
# Returns list of lists: [[tok_count_variant_0, tok_count_variant_1, ...], ...]
answer_token_counts = []
for answers in all_queries["answers"]:
    counts = [len(tokenizer.encode(a, add_special_tokens=False)) for a in answers]
    answer_token_counts.append(counts)

# --- Distribution overview ---
# Flatten all variant counts for global stats
all_counts = [c for counts in answer_token_counts for c in counts]
print("Token count distribution (all answer variants, Contriever tokenizer):")
print(pd.Series(all_counts).describe())
print(f"\nHistogram (token counts 1-15):")
print(pd.Series(all_counts).value_counts().sort_index().head(15))

Token count distribution (all answer variants, Contriever tokenizer):
count    101855.000000
mean          2.987227
std           1.672249
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          23.000000
dtype: float64

Histogram (token counts 1-15):
1     20932
2     24850
3     20315
4     19835
5      8697
6      3657
7      1954
8       894
9       403
10      176
11       81
12       34
13       10
14        7
15        2
Name: count, dtype: int64


In [5]:
# --- Split analysis: queries with mixed pass/fail across answer variants ---
# A query has "split answers" when some variants are <= threshold and others > threshold.
# This tells us whether filtering on ALL vs ANY variant matters.

n_all_pass = 0    # all variants <= 5
n_all_fail = 0    # all variants > 5
n_split = 0       # some pass, some fail
split_examples = []

for i, counts in enumerate(answer_token_counts):
    passes = [c <= TOKEN_THRESHOLD for c in counts]
    if all(passes):
        n_all_pass += 1
    elif not any(passes):
        n_all_fail += 1
    else:
        n_split += 1
        if len(split_examples) < 10:
            answers = all_queries[i]["answers"]
            split_examples.append({
                "question": all_queries[i]["question"],
                "answers": list(zip(answers, counts)),
            })

total = len(answer_token_counts)
print(f"Token threshold: <= {TOKEN_THRESHOLD} (Contriever tokenizer)\n")
print(f"All variants pass:   {n_all_pass:,} ({n_all_pass/total*100:.1f}%)")
print(f"All variants fail:   {n_all_fail:,} ({n_all_fail/total*100:.1f}%)")
print(f"Split (some pass):   {n_split:,} ({n_split/total*100:.1f}%)")
print(f"Total:               {total:,}")

if split_examples:
    print(f"\n--- Examples of split answers (answer, token_count) ---")
    for ex in split_examples[:5]:
        print(f"\nQ: {ex['question']}")
        for ans, tc in ex["answers"]:
            marker = "PASS" if tc <= TOKEN_THRESHOLD else "FAIL"
            print(f"  [{marker}] \"{ans}\" -> {tc} tokens")

Token threshold: <= 5 (Contriever tokenizer)

All variants pass:   76,406 (91.9%)
All variants fail:   5,494 (6.6%)
Split (some pass):   1,204 (1.4%)
Total:               83,104

--- Examples of split answers (answer, token_count) ---

Q: who is the main character in memoirs of a geisha
  [FAIL] "`` Sayuri ''" -> 6 tokens
  [PASS] "Chiyo Sakamoto" -> 5 tokens

Q: who played the brothers in seven brides for seven brothers
  [PASS] "Tommy Rall" -> 3 tokens
  [PASS] "Howard Keel" -> 2 tokens
  [PASS] "Jeff Richards" -> 2 tokens
  [PASS] "Russ Tamblyn" -> 4 tokens
  [PASS] "Matt Mattox" -> 3 tokens
  [FAIL] "Jacques d'Amboise" -> 6 tokens
  [PASS] "Marc Platt" -> 2 tokens

Q: who died at the gunfight at okay corral
  [FAIL] "Tom and Frank McLaury" -> 6 tokens
  [PASS] "Billy Clanton" -> 3 tokens

Q: where did they shoot butch cassidy and the sundance kid
  [PASS] "Utah" -> 1 tokens
  [PASS] "Colorado" -> 1 tokens
  [FAIL] "Cuernavaca and Taxco , Mexico" -> 9 tokens
  [PASS] "New Mexico" ->

In [6]:
# --- Apply token count filter: keep only queries where ALL answer variants <= 5 tokens ---
# Why ALL (not ANY): conservative approach, ensures every ground-truth answer
# is a short factoid entity — consistent with the proposal's intent.
# Cost: only 1.4% of queries lost vs ANY strategy.

passing_indices = [
    i for i, counts in enumerate(answer_token_counts)
    if all(c <= TOKEN_THRESHOLD for c in counts)
]

filtered_by_tokens = all_queries.select(passing_indices)

print(f"Before token filter: {len(all_queries):,}")
print(f"After token filter:  {len(filtered_by_tokens):,} ({len(filtered_by_tokens)/len(all_queries)*100:.1f}%)")
print(f"Dropped:             {len(all_queries) - len(filtered_by_tokens):,}")

# Quick sanity check
print(f"\nSample filtered query:")
print(f"  Q: {filtered_by_tokens[0]['question']}")
print(f"  A: {filtered_by_tokens[0]['answers']}")

Before token filter: 83,104
After token filter:  76,406 (91.9%)
Dropped:             6,698

Sample filtered query:
  Q: total number of death row inmates in the us
  A: ['2,718']


## Step 2 — Entity linking with ReFiNed

**Goal**: extract Wikidata entities (QIDs) from both questions and answers using ReFiNed.

**Model choice**: `questions_model` — fine-tuned on WebQSP entity linking dataset,
optimized for question-style text (lowercase, no punctuation). Better fit than
`wikipedia_model` which is trained on well-formed Wikipedia prose.

**Entity set**: `wikipedia` (~6M entities) — covers entities with Wikipedia pages,
which is sufficient for NQ-open (all answers come from Wikipedia). Returns Wikidata QIDs.
If coverage is insufficient, we can upgrade to `wikidata` (~33M entities, ~20 GB download).

**Filter rule** (from proposal): keep only queries where **both** question and answer
contain at least one recognized Wikidata entity.

**Compatibility patches** (ReFiNed V1 on Windows / Python 3.12+ / transformers 4.48+) —
applied via `python scripts/patch_refined.py` (re-run after `uv sync` / reinstall):
1. `aws.py` — `strftime("%s")` → `.timestamp()` (Unix-only, 0-byte downloads on Windows)
2. `loaders.py` — `re.compile(...)` → raw string (invalid escape, SyntaxWarning)
3. `general_utils.py` — drop `add_special_tokens` kwarg from `from_pretrained()` (conflicts with tokenizer method in transformers >= 4.x)

In [7]:
import os
import subprocess
import sys
from pathlib import Path

# =============================================================================
# ReFiNed V1 compatibility - source-level patches
#
# Patches are applied automatically by running scripts/patch_refined.py.
# This cell invokes the script so patches are always up-to-date, even after
# uv sync / reinstall.
#
# 1. aws.py - strftime("%s") -> .timestamp() (Unix-only on Windows)
# 2. loaders.py - re.compile() -> raw string (SyntaxWarning Python 3.12+)
# 3. general_utils.py - drop add_special_tokens kwarg (transformers >= 4.x)
# 4. data_lookups.py - drop add_special_tokens kwarg (transformers >= 4.x)
# =============================================================================

subprocess.check_call([sys.executable, "scripts/patch_refined.py"])

# --- Local model cache ---
REFINED_DATA_DIR = Path.cwd() / "data" / "refined_cache"
REFINED_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nReFiNed cache dir: {REFINED_DATA_DIR}")


ReFiNed cache dir: C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\data\refined_cache


In [8]:

from refined.inference.processor import Refined

# --- Load ReFiNed model ---
# questions_model: fine-tuned on WebQSP for question-style text
# entity_set="wikipedia": ~6M entities, sufficient for NQ-open
# data_dir: local cache in data/refined_cache/ (gitignored, persists across sessions)
# First run downloads model weights from S3; subsequent runs load from local cache.
refined_model = Refined.from_pretrained(
    model_name="questions_model",
    entity_set="wikipedia",
    data_dir=str(REFINED_DATA_DIR),
)
print("ReFiNed model loaded")

# --- Quick test ---
test_spans = refined_model.process_text("who owned the millennium falcon before han solo")
print("\nTest entity linking:")
for span in test_spans:
    ent = span.predicted_entity
    qid = ent.wikidata_entity_id if ent else None
    title = ent.wikipedia_entity_title if ent else None
    print(f"  \"{span.text}\" -> {qid} ({title})")

ReFiNed model loaded


C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\.venv\Lib\site-packages\refined\inference\processor.py:293: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Test entity linking:
  "millennium falcon" -> Q19901 (Millennium Falcon)
  "han" -> Q490452 (Han Suk-kyu)
